In [1]:
import sys
from pathlib import Path

# Dodaj repo root do sys.path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"Python version: {sys.version}")

REPO_ROOT: /run/media/mwrona/Nowy/Engineering-Thesis
Python version: 3.14.2 (main, Dec  5 2025, 00:00:00) [GCC 15.2.1 20251111 (Red Hat 15.2.1-4)]


## 1. Import torch i podstawowe info

In [2]:
try:
    import torch
    print("✓ torch zaimportowany pomyślnie")
    print(f"  Wersja: {torch.__version__}")
    print(f"  Lokalizacja: {torch.__file__}")
except ImportError as e:
    print(f"✗ Błąd importu torch: {e}")
    print("  Sprawdź czy torch jest zainstalowany: pip install torch")

✓ torch zaimportowany pomyślnie
  Wersja: 2.9.1+cu128
  Lokalizacja: /run/media/mwrona/Nowy/Engineering-Thesis/.venv/lib64/python3.14/site-packages/torch/__init__.py


## 2. Sprawdzenie dostępności CUDA

In [3]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA dostępna: {cuda_available}")

if cuda_available:
    print(f"  Liczba urządzeń CUDA: {torch.cuda.device_count()}")
    print(f"  Aktualne urządzenie: {torch.cuda.current_device()}")
    print(f"  Nazwa urządzenia: {torch.cuda.get_device_name(0)}")
    print(f"  Wersja CUDA: {torch.version.cuda}")
    print(f"  cuDNN dostępny: {torch.backends.cudnn.is_available()}")
    if torch.backends.cudnn.is_available():
        print(f"  Wersja cuDNN: {torch.backends.cudnn.version()}")
else:
    print("  → Obliczenia będą wykonywane na CPU")

# Zalecane urządzenie
device = 'cuda' if cuda_available else 'cpu'
print(f"\nZalecane urządzenie: {device}")

CUDA dostępna: True
  Liczba urządzeń CUDA: 1
  Aktualne urządzenie: 0
  Nazwa urządzenia: NVIDIA GeForce RTX 3080 Laptop GPU
  Wersja CUDA: 12.8
  cuDNN dostępny: True
  Wersja cuDNN: 91002

Zalecane urządzenie: cuda


## 3. Podstawowe operacje na tensorach (CPU)

In [4]:
import torch
import numpy as np

print("=== Test 1: Tworzenie tensorów ===")
t1 = torch.tensor([1.0, 2.0, 3.0])
t2 = torch.zeros((3, 3))
t3 = torch.randn(2, 4)
print(f"✓ t1: {t1.shape}, dtype={t1.dtype}, device={t1.device}")
print(f"✓ t2: {t2.shape}, dtype={t2.dtype}, device={t2.device}")
print(f"✓ t3: {t3.shape}, dtype={t3.dtype}, device={t3.device}")

print("\n=== Test 2: Operacje arytmetyczne ===")
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
c = a + b
d = torch.matmul(a, b)
print(f"✓ Dodawanie: {c.shape}")
print(f"✓ Mnożenie macierzy: {d.shape}")
print(f"  d =\n{d}")

print("\n=== Test 3: Konwersja numpy <-> torch ===")
np_array = np.array([[1, 2], [3, 4]], dtype=np.float32)
torch_tensor = torch.from_numpy(np_array)
back_to_np = torch_tensor.numpy()
print(f"✓ NumPy -> Torch: {torch_tensor.shape}, dtype={torch_tensor.dtype}")
print(f"✓ Torch -> NumPy: {back_to_np.shape}, dtype={back_to_np.dtype}")
print(f"✓ Równość: {np.allclose(np_array, back_to_np)}")

=== Test 1: Tworzenie tensorów ===
✓ t1: torch.Size([3]), dtype=torch.float32, device=cpu
✓ t2: torch.Size([3, 3]), dtype=torch.float32, device=cpu
✓ t3: torch.Size([2, 4]), dtype=torch.float32, device=cpu

=== Test 2: Operacje arytmetyczne ===
✓ Dodawanie: torch.Size([2, 2])
✓ Mnożenie macierzy: torch.Size([2, 2])
  d =
tensor([[19., 22.],
        [43., 50.]])

=== Test 3: Konwersja numpy <-> torch ===
✓ NumPy -> Torch: torch.Size([2, 2]), dtype=torch.float32
✓ Torch -> NumPy: (2, 2), dtype=float32
✓ Równość: True


## 4. Test operacji na CUDA (jeśli dostępna)

In [5]:
import torch

if torch.cuda.is_available():
    print("=== Test CUDA ===")
    
    # Tworzenie tensorów na GPU
    t_cpu = torch.randn(3, 3)
    t_gpu = t_cpu.to('cuda')
    print(f"✓ Tensor na CPU: device={t_cpu.device}")
    print(f"✓ Tensor na GPU: device={t_gpu.device}")
    
    # Operacje na GPU
    a_gpu = torch.randn(1000, 1000, device='cuda')
    b_gpu = torch.randn(1000, 1000, device='cuda')
    c_gpu = torch.matmul(a_gpu, b_gpu)
    print(f"✓ Mnożenie macierzy na GPU: {c_gpu.shape}, device={c_gpu.device}")
    
    # Transfer GPU -> CPU
    c_cpu = c_gpu.cpu()
    print(f"✓ Transfer GPU->CPU: device={c_cpu.device}")
    
    print("\n✓ Wszystkie operacje CUDA przeszły pomyślnie")
else:
    print("⊘ CUDA niedostępna, pomijam test GPU")

=== Test CUDA ===
✓ Tensor na CPU: device=cpu
✓ Tensor na GPU: device=cuda:0
✓ Mnożenie macierzy na GPU: torch.Size([1000, 1000]), device=cuda:0
✓ Transfer GPU->CPU: device=cpu

✓ Wszystkie operacje CUDA przeszły pomyślnie


## 5. Test torch.multinomial (kluczowe dla samplingów)

In [6]:
import torch

print("=== Test torch.multinomial ===")

# CPU
probs_cpu = torch.tensor([0.1, 0.2, 0.3, 0.4])
samples_cpu = torch.multinomial(probs_cpu, num_samples=10, replacement=True)
print(f"✓ Sampling na CPU: {samples_cpu}")
print(f"  Unikalne wartości: {torch.unique(samples_cpu).tolist()}")

# GPU (jeśli dostępna)
if torch.cuda.is_available():
    probs_gpu = probs_cpu.to('cuda')
    samples_gpu = torch.multinomial(probs_gpu, num_samples=10, replacement=True)
    print(f"✓ Sampling na GPU: {samples_gpu}")
    print(f"  Unikalne wartości: {torch.unique(samples_gpu).tolist()}")

# Test z generatorem (dla deterministycznego seedowania)
gen = torch.Generator()
gen.manual_seed(42)
samples_seeded = torch.multinomial(probs_cpu, num_samples=5, replacement=True, generator=gen)
print(f"✓ Sampling z seedem: {samples_seeded}")

# Drugi raz z tym samym seedem (powinno być inne, generator się przesuwa)
samples_seeded2 = torch.multinomial(probs_cpu, num_samples=5, replacement=True, generator=gen)
print(f"✓ Kolejny sampling: {samples_seeded2}")

# Reset generatora
gen.manual_seed(42)
samples_reset = torch.multinomial(probs_cpu, num_samples=5, replacement=True, generator=gen)
print(f"✓ Po resecie (seed=42): {samples_reset}")
print(f"  Zgodność z pierwszym: {torch.equal(samples_seeded, samples_reset)}")

=== Test torch.multinomial ===
✓ Sampling na CPU: tensor([3, 1, 2, 2, 3, 2, 2, 3, 3, 1])
  Unikalne wartości: [1, 2, 3]
✓ Sampling na GPU: tensor([1, 0, 1, 3, 3, 1, 1, 3, 3, 2], device='cuda:0')
  Unikalne wartości: [0, 1, 2, 3]
✓ Sampling z seedem: tensor([0, 0, 1, 0, 2])
✓ Kolejny sampling: tensor([2, 3, 3, 0, 1])
✓ Po resecie (seed=42): tensor([0, 0, 1, 0, 2])
  Zgodność z pierwszym: True


## 6. Test importu naszych algorytmów torch

In [7]:
print("=== Test importu algorytmów torch z repo ===")

try:
    from src.algorithms import torch_utils
    print("✓ torch_utils zaimportowany")
    print(f"  TORCH_AVAILABLE: {torch_utils.TORCH_AVAILABLE}")
    print(f"  get_device(): {torch_utils.get_device()}")
except ImportError as e:
    print(f"✗ Błąd importu torch_utils: {e}")

try:
    from src.algorithms import qh_qlearning_torch
    print("✓ qh_qlearning_torch zaimportowany")
    print(f"  Klasa: {qh_qlearning_torch.QHQLearningTorch}")
except ImportError as e:
    print(f"✗ Błąd importu qh_qlearning_torch: {e}")

try:
    from src.algorithms import qh_policy_evaluation_torch
    print("✓ qh_policy_evaluation_torch zaimportowany")
    print(f"  Klasa: {qh_policy_evaluation_torch.QHPolicyEvaluationTorch}")
except ImportError as e:
    print(f"✗ Błąd importu qh_policy_evaluation_torch: {e}")

=== Test importu algorytmów torch z repo ===
✓ torch_utils zaimportowany


AttributeError: module 'src.algorithms.torch_utils' has no attribute 'TORCH_AVAILABLE'

## 7. Mini-test: Prosty MDP z torch backend

In [8]:
import torch
import numpy as np

print("=== Mini-test: 2-stanowy MDP z torch ===")

# Parametry
n_states = 2
n_actions = 2
alpha = 0.5
beta = 0.9

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Urządzenie: {device}")

# Macierze przejść P[s,a,s'] i nagród R[s,a]
P = np.zeros((n_states, n_actions, n_states), dtype=np.float32)
P[0, 0, 1] = 1.0  # s=0, a=0 -> deterministycznie do s=1
P[0, 1, :] = 0.5  # s=0, a=1 -> po 50% do s=0 i s=1
P[1, 0, :] = 0.5  # s=1, a=0 -> po 50%
P[1, 1, :] = 0.5  # s=1, a=1 -> po 50%

R = np.array([[0.0, 2.0],
              [15.0, 10.0]], dtype=np.float32)

# Konwersja do torch
P_t = torch.from_numpy(P).to(device)
R_t = torch.from_numpy(R).to(device)

print(f"✓ P tensor: shape={P_t.shape}, device={P_t.device}, dtype={P_t.dtype}")
print(f"✓ R tensor: shape={R_t.shape}, device={R_t.device}, dtype={R_t.dtype}")

# Inicjalizacja wartości
Q = torch.zeros((n_states, n_actions), device=device, dtype=torch.float32)
W = torch.zeros((n_states, n_actions), device=device, dtype=torch.float32)

print(f"✓ Q tensor: shape={Q.shape}, device={Q.device}")
print(f"✓ W tensor: shape={W.shape}, device={W.device}")

# Prosta aktualizacja (pseudokod TD)
s, a = 0, 0
s_next = 1
r = R_t[s, a].item()
eta = 0.1
theta = 0.05

# W update (beta-only)
target_W = r + beta * W[s_next, :].max().item()
W[s, a] = W[s, a] + theta * (target_W - W[s, a])

# Q update (alpha*beta)
target_Q = r + alpha * beta * W[s_next, :].max().item()
Q[s, a] = Q[s, a] + eta * (target_Q - Q[s, a])

print(f"\n✓ Po jednej aktualizacji (s={s}, a={a}):")
print(f"  W[{s},{a}] = {W[s, a].item():.4f}")
print(f"  Q[{s},{a}] = {Q[s, a].item():.4f}")

# Transfer z powrotem do CPU/NumPy
Q_np = Q.cpu().numpy()
W_np = W.cpu().numpy()
print(f"\n✓ Konwersja do NumPy:")
print(f"  Q shape: {Q_np.shape}, dtype: {Q_np.dtype}")
print(f"  W shape: {W_np.shape}, dtype: {W_np.dtype}")

print("\n✓ Mini-test zakończony pomyślnie")

=== Mini-test: 2-stanowy MDP z torch ===
Urządzenie: cuda
✓ P tensor: shape=torch.Size([2, 2, 2]), device=cuda:0, dtype=torch.float32
✓ R tensor: shape=torch.Size([2, 2]), device=cuda:0, dtype=torch.float32
✓ Q tensor: shape=torch.Size([2, 2]), device=cuda:0
✓ W tensor: shape=torch.Size([2, 2]), device=cuda:0

✓ Po jednej aktualizacji (s=0, a=0):
  W[0,0] = 0.0000
  Q[0,0] = 0.0000

✓ Konwersja do NumPy:
  Q shape: (2, 2), dtype: float32
  W shape: (2, 2), dtype: float32

✓ Mini-test zakończony pomyślnie


## 8. Podsumowanie

In [9]:
import torch

print("=" * 60)
print("PODSUMOWANIE TESTÓW TORCH")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
print(f"Zalecane urządzenie: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("\n✓ Wszystkie podstawowe testy przeszły pomyślnie")
print("✓ Środowisko gotowe do eksperymentów z QH algorytmami")

PODSUMOWANIE TESTÓW TORCH
PyTorch version: 2.9.1+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3080 Laptop GPU
CUDA version: 12.8
Zalecane urządzenie: cuda

✓ Wszystkie podstawowe testy przeszły pomyślnie
✓ Środowisko gotowe do eksperymentów z QH algorytmami
